# 03 — Features, Model & Honest Backtest

Goal: beat the seasonal-naive baseline from notebook 02, without fooling
ourselves. Everything here calls the production code in `src/`, so what is
measured in this notebook is exactly what runs in the deployed service.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)


In [ ]:
from src.pipeline import FEATURE_COLS, encode_categories
from src.forecast import (
    ALL_FEATURES, build_supervised, make_model, rolling_origin_backtest,
    seasonal_naive, wape, bias,
)

weekly = pd.read_csv(C.WEEKLY_PATH, parse_dates=["week_start"])
features = pd.read_csv(C.FEATURES_PATH, parse_dates=["week_start", "launch_date"])
sup = build_supervised(encode_categories(features))
print(f"Supervised rows: {len(sup):,}   features: {len(ALL_FEATURES)}")

## 1. The leakage rule

A feature for week *w* may only use data from before *w*. Verify it directly
rather than trusting the code comment.

In [ ]:
one = features[features["sku_id"] == features["sku_id"].iloc[0]].sort_values("week_start")
check = one[["week_start", "units_sold", "lag_1", "roll_mean_4"]].head(8)
check["manual_lag_1"] = one["units_sold"].shift(1).head(8)
check["manual_roll_4"] = one["units_sold"].shift(1).rolling(4, min_periods=2).mean().head(8)
check.set_index("week_start").round(2)

`lag_1` matches the previous week's actual and `roll_mean_4` is shifted by one
— neither touches the target week. The automated version of this check lives in
`tests/test_pipeline.py` and runs on every change.

## 2. Why direct multi-horizon, not recursive

A recursive forecaster predicts week 1, feeds that prediction back in as a lag,
and predicts week 2 — so by week 8 it is mostly reading its own guesses, and
errors compound. Here every row carries the horizon *h* as a feature and is
trained against genuine observed demand *h* weeks after a real origin.

In [ ]:
sup[["sku_id", "origin_week", "target_week", "h", "y", "lag_1", "target_lag_52"]].head(10)

## 3. Rolling-origin backtest

Never a random split. A random split lets the model train on the future of the
same series it is tested on, which inflates accuracy in a way that disappears in
production.

In [ ]:
bt = rolling_origin_backtest(sup, weekly)
folds = pd.DataFrame(bt["folds"])
folds[["fold", "train_until", "test_from", "test_to", "model_wape", "baseline_wape", "model_bias"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(len(folds))
ax.bar(x - 0.2, folds["model_wape"], 0.4, label="Model", color="#6b5bd6")
ax.bar(x + 0.2, folds["baseline_wape"], 0.4, label="Seasonal-naive", color="#d69b45")
ax.set_xticks(x); ax.set_xticklabels(folds["fold"])
ax.set_xlabel("Fold"); ax.set_ylabel("WAPE"); ax.legend()
ax.set_title("Model vs baseline, per backtest fold (lower is better)")
plt.show()

print(f"Mean model WAPE    : {bt['mean_model_wape']:.4f}")
print(f"Mean baseline WAPE : {bt['mean_baseline_wape']:.4f}")
print(f"Improvement        : {bt['improvement_vs_baseline_pct']:.1f}%")
print(f"Beats baseline     : {bt['model_beats_baseline']}")

### The mistake worth recording

The first version of this model beat the baseline by only 1.5% and *lost* one
fold. The cause was a real modelling error: the model received `lag_52` relative
to the **origin** week, while the baseline used demand from the same week last
year relative to the **target** week. The model was being asked to beat a
benchmark while denied the benchmark's own information.

Adding `target_lag_52` (and a smoothed 3-week version, plus its ratio to the
current level) took the improvement to ~20% with wins in every fold. This is
recorded rather than quietly overwritten, because knowing *why* a model improved
is the difference between a result and a lucky number.

## 4. Which features carry the forecast

Permutation importance on a holdout — slower than a built-in attribute, but it
measures what the model actually relies on for out-of-sample accuracy.

In [ ]:
from sklearn.inspection import permutation_importance

labelled = sup[sup["y"].notna() & sup["roll_mean_4"].notna()]
cut = labelled["target_week"].max() - pd.Timedelta(weeks=C.HORIZON_WEEKS)
tr, te = labelled[labelled["target_week"] < cut], labelled[labelled["target_week"] >= cut]

m = make_model()
m.fit(tr[ALL_FEATURES], tr["y"].clip(lower=0))

sample = te.sample(min(3000, len(te)), random_state=42)
imp = permutation_importance(
    m, sample[ALL_FEATURES], sample["y"], n_repeats=3, random_state=42, scoring="neg_mean_absolute_error"
)
pd.DataFrame({"feature": ALL_FEATURES, "importance": imp.importances_mean}) \
    .sort_values("importance", ascending=False).head(15).round(4)

## 5. Accuracy by horizon

Forecasts get worse further out. The ops team needs to know by how much, because
a week-8 number should be trusted less than a week-1 number.

In [ ]:
te_pred = te.assign(pred=np.clip(m.predict(te[ALL_FEATURES]), 0, None))
by_h = te_pred.groupby("h").apply(
    lambda g: pd.Series({
        "model_wape": wape(g["y"].to_numpy(), g["pred"].to_numpy()),
        "bias": bias(g["y"].to_numpy(), g["pred"].to_numpy()),
    }),
    include_groups=False,
)
print(by_h.round(4))

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(by_h.index, by_h["model_wape"], marker="o", color="#6b5bd6")
ax.set_xlabel("Weeks ahead"); ax.set_ylabel("WAPE")
ax.set_title("Forecast error grows with horizon")
plt.show()

## 6. Handing off

`python run_all.py` reproduces all of the above and writes the artefacts the
dashboard and the scoring service read. Risk scoring is deliberately rule-based
and lives in `src/risk.py` — see the README for the definitions.